# Silver Layer — Tabele de referință (Nomenclatoare)

**Scop:** Curățarea și validarea celor 14 tabele `ref_*` din Bronze și încărcarea lor în `banking.silver.ref_*`.

**De ce sunt simple aceste tabele:**
- Datele sunt **statice / semi-statice** — nu se schimbă des
- Sunt seed-uite manual din `seed_nomenclatoare.sql` — nu conțin erori injectate
- Validările sunt minime: PK non-null, valori unice, type casting

**Transformări:**
1. Cast STRING → tipuri reale (INTEGER, REAL, DATE)
2. Validare: PK non-null și unic
3. Eliminăm coloanele de metadata Bronze (`_ingestion_ts`, etc.) — nu mai sunt necesare în Silver
4. Adăugăm `_silver_loaded_at` pentru audit Silver

**Strategie scriere:** `overwrite` — refresh complet la fiecare rulare (datele sunt mici).

## 1. Configurare

In [0]:
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType

CATALOG = "banking"
BRONZE  = "bronze"
SILVER  = "silver"

print(f"Citim din: {CATALOG}.{BRONZE}.raw_ref_*")
print(f"Scriem in: {CATALOG}.{SILVER}.ref_*")

Citim din: banking.bronze.raw_ref_*
Scriem in: banking.silver.ref_*


## 2. Configurația de transformare per tabel

Pentru fiecare tabel `ref_*`, definim:
- **`pk`**: cheia primară (pentru validare unicitate)
- **`int_cols`**: coloane care trebuie convertite la INTEGER (flag-uri 0/1, scoruri etc.)
- **`double_cols`**: coloane care trebuie convertite la REAL (rate, sume, limite)

Coloanele rămase rămân STRING (sunt deja text natural).

In [0]:
REF_TABLES_CONFIG = {
    "ref_countries": {
        "pk": "country_code",
        "int_cols": ["is_eu", "is_sepa", "is_active"],
    },
    "ref_counties": {
        "pk": "county_code",
        "int_cols": ["is_active"],
    },
    "ref_regions": {
        "pk": "region_code",
        "int_cols": ["counties_count"],
    },
    "ref_currencies": {
        "pk": "currency_code",
        "int_cols": ["decimals", "is_base", "is_active"],
        "double_cols": ["exchange_rate"],
        "drop_cols": ["symbol"],
    },
    "ref_transaction_types": {
        "pk": "type_code",
        "int_cols": ["affects_balance", "requires_card"],
    },
    "ref_transaction_statuses": {
        "pk": "status_code",
        "int_cols": ["is_final", "retry_allowed"],
    },
    "ref_customer_segments": {
        "pk": "segment_code",
        "int_cols": ["fee_waiver", "priority_support"],
        "double_cols": ["min_balance"],
    },
    "ref_account_types": {
        "pk": "type_code",
        "int_cols": ["allows_overdraft", "taxable"],
        "double_cols": ["min_balance", "interest_rate_pa"],
    },
    "ref_loan_types": {
        "pk": "loan_type_code",
        "int_cols": ["max_term_months", "requires_collateral"],
        "double_cols": ["max_ltv", "typical_rate_pa"],
    },
    "ref_card_types": {
        "pk": "card_type_code",
        "int_cols": ["has_credit_line", "contactless"],
        "double_cols": ["annual_fee"],
    },
    "ref_channels": {
        "pk": "channel_code",
        "int_cols": ["requires_pin", "is_active"],
        "double_cols": ["max_tx_amount", "max_daily_amount"],
    },
    "ref_risk_bands": {
        "pk": "band_code",
        "int_cols": ["score_min", "score_max", "auto_approve", "auto_reject"],
    },
    "ref_kyc_statuses": {
        "pk": "status_code",
        "int_cols": ["requires_review", "aml_flag", "allows_transactions"],
    },
    "ref_employee_roles": {
        "pk": "role_code",
        "int_cols": ["can_approve_loans", "can_block_accounts", "can_view_reports"],
        "double_cols": ["loan_approval_limit"],
    },
}

print(f"Configurate {len(REF_TABLES_CONFIG)} tabele de referinta")

Configurate 14 tabele de referinta


## 3. Funcție generică de transformare

Aplică type casting, validări și scriere pentru orice tabel ref.

In [0]:
def transform_ref_table(
    table_name: str,
    config: dict,
) -> dict:
    """
    Citeste raw_<table_name> din Bronze, aplica transformari si scrie in Silver.
    Returneaza statistici despre operatie.
    """
    start = datetime.now()
    
    # Citire din Bronze
    bronze_table = f"{CATALOG}.{BRONZE}.raw_{table_name}"
    df = spark.table(bronze_table)
    rows_in = df.count()
    
    # 1. Pastram metadata Bronze redenumita pentru lineage
    if "_batch_id" in df.columns:
        df = df.withColumnRenamed("_batch_id", "source_batch_id")
    if "_ingestion_ts" in df.columns:
        df = df.withColumnRenamed("_ingestion_ts", "source_ingestion_ts")

    # Eliminam doar metadatele duplicate / mai puțin utile
    for c in ["_source_system", "_source_file"]:
        if c in df.columns:
            df = df.drop(c)

    # 1b. Eliminam coloane cosmetice / inutile pentru analiza
    drop_cols = config.get("drop_cols", [])
    existing_drops = [c for c in drop_cols if c in df.columns]
    if existing_drops:
        df = df.drop(*existing_drops)
    
    # 2. Type casting INTEGER
    for c in config.get("int_cols", []):
        if c in df.columns:
            df = df.withColumn(c, col(c).cast(IntegerType()))
    
    # 3. Type casting DOUBLE
    for c in config.get("double_cols", []):
        if c in df.columns:
            df = df.withColumn(c, col(c).cast(DoubleType()))
    
    # 4. Validare: PK non-null
    pk = config["pk"]
    df_valid = df.filter(col(pk).isNotNull() & (col(pk) != "None"))
    rows_valid = df_valid.count()
    rows_dropped = rows_in - rows_valid
    
    # 5. Verificare unicitate PK (warning, nu eroare)
    distinct_pk = df_valid.select(pk).distinct().count()
    has_duplicates = distinct_pk != rows_valid
    
    # 6. Adaugam timestamp Silver
    df_final = df_valid.withColumn("silver_loaded_at", current_timestamp())
    
    # 7. Scriem in Silver — overwrite
    target_table = f"{CATALOG}.{SILVER}.{table_name}"
    (df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table))
    
    elapsed = (datetime.now() - start).total_seconds()
    
    return {
        "table"          : table_name,
        "rows_in"        : rows_in,
        "rows_valid"     : rows_valid,
        "rows_dropped"   : rows_dropped,
        "has_duplicates" : has_duplicates,
        "elapsed_s"      : round(elapsed, 2),
    }

## 4. Rulare pentru toate tabelele ref_*

In [0]:
results = []
total_start = datetime.now()

print(f"{'Tabel':30s} {'In':>8s} {'Valid':>8s} {'Drop':>6s} {'Dup?':>6s} {'Timp(s)':>8s}")
print("-" * 75)

for table_name, config in REF_TABLES_CONFIG.items():
    try:
        r = transform_ref_table(table_name, config)
        results.append(r)
        dup_marker = "DA" if r["has_duplicates"] else "-"
        print(
            f"{r['table']:30s} "
            f"{r['rows_in']:>8d} "
            f"{r['rows_valid']:>8d} "
            f"{r['rows_dropped']:>6d} "
            f"{dup_marker:>6s} "
            f"{r['elapsed_s']:>8.2f}"
        )
    except Exception as e:
        print(f"  EROARE la {table_name}: {e}")
        results.append({"table": table_name, "error": str(e)})

total_elapsed = (datetime.now() - total_start).total_seconds()
print("-" * 75)
print(f"Total: {len(results)} tabele | {total_elapsed:.1f}s")

# Verificam daca avem erori sau drops
errors = [r for r in results if "error" in r]
dropped_total = sum(r.get("rows_dropped", 0) for r in results)
duplicates = [r for r in results if r.get("has_duplicates")]

if errors:
    print(f"\nERORI: {len(errors)} tabele au esuat")
if dropped_total > 0:
    print(f"\nWARNING: {dropped_total} randuri eliminate (PK null)")
if duplicates:
    print(f"\nWARNING: {len(duplicates)} tabele au PK-uri duplicate")
if not errors and dropped_total == 0 and not duplicates:
    print("\nTOATE TABELELE INCARCATE CURAT")

Tabel                                In    Valid   Drop   Dup?  Timp(s)
---------------------------------------------------------------------------
ref_countries                        20       20      0      -     4.68
ref_counties                         42       42      0      -     3.59
ref_regions                           8        8      0      -     3.88
ref_currencies                        6        6      0      -     5.22
ref_transaction_types                13       13      0      -     3.99
ref_transaction_statuses              5        5      0      -     3.71
ref_customer_segments                 4        4      0      -     3.96
ref_account_types                     4        4      0      -     4.10
ref_loan_types                        6        6      0      -     3.91
ref_card_types                        6        6      0      -     3.93
ref_channels                          7        7      0      -     4.21
ref_risk_bands                        4        4      0     

## 5. Verificare — listare tabele Silver create

In [0]:
%sql
SHOW TABLES IN banking.silver;

database,tableName,isTemporary
silver,dim_accounts,false
silver,dim_branches,false
silver,dim_cards,false
silver,dim_employees,false
silver,dim_loans,false
silver,ref_account_types,false
silver,ref_card_types,false
silver,ref_channels,false
silver,ref_counties,false
silver,ref_countries,false


## 6. Verificare schema unei tabele transformate

Comparăm `raw_currencies` (Bronze, toate STRING) cu `ref_currencies` (Silver, tipuri corecte).

In [0]:
%sql
-- Schema Bronze (toate STRING)
DESCRIBE banking.bronze.raw_ref_currencies;

col_name,data_type,comment
currency_code,string,null
name_ro,string,null
symbol,string,null
decimals,string,null
is_base,string,null
exchange_rate,string,null
is_active,string,null
updated_at,string,null
_ingestion_ts,timestamp,null
_source_system,string,null


In [0]:
%sql
-- Schema Silver (tipuri corecte: INT, DOUBLE)
DESCRIBE banking.silver.ref_currencies;

col_name,data_type,comment
currency_code,string,null
name_ro,string,null
decimals,int,null
is_base,int,null
exchange_rate,double,null
is_active,int,null
updated_at,string,null
source_ingestion_ts,timestamp,null
source_batch_id,string,null
silver_loaded_at,timestamp,null


## 7. Inspecție date — ref_currencies după transformare

In [0]:
%sql
SELECT 
    currency_code, 
    name_ro,  
    decimals,        -- INTEGER acum
    exchange_rate,   -- DOUBLE acum
    is_base,
    silver_loaded_at
FROM banking.silver.ref_currencies
ORDER BY currency_code;

currency_code,name_ro,decimals,exchange_rate,is_base,silver_loaded_at
CHF,Franc elvetian,2,5.1,0,2026-05-01T19:41:03.896Z
EUR,Euro,2,4.97,0,2026-05-01T19:41:03.896Z
GBP,Lira sterlina,2,5.83,0,2026-05-01T19:41:03.896Z
HUF,Forint maghiar,0,0.0126,0,2026-05-01T19:41:03.896Z
RON,Leu romanesc,2,1.0,1,2026-05-01T19:41:03.896Z
USD,Dolar american,2,4.61,0,2026-05-01T19:41:03.896Z


## 8. Test: putem face calcule numerice corect

Demonstrăm că type casting-ul a funcționat — putem face matematici pe `exchange_rate`.

In [0]:
%sql
-- Calcul: 1000 unitati din fiecare valuta in RON
SELECT 
    currency_code,
    exchange_rate,
    1000 * exchange_rate AS valoare_in_ron
FROM banking.silver.ref_currencies
WHERE is_active = 1
ORDER BY exchange_rate DESC;

currency_code,exchange_rate,valoare_in_ron
GBP,5.83,5830.0
CHF,5.1,5100.0
EUR,4.97,4970.0
USD,4.61,4610.0
RON,1.0,1000.0
HUF,0.0126,12.6


## Sumar Silver References

Ce am realizat:
- Citit cele 14 tabele `raw_ref_*` din Bronze
- Aplicat **type casting** corect (STRING → INTEGER, DOUBLE)
- Validat PK non-null pe fiecare tabelă
- Eliminat coloanele de metadata Bronze
- Adăugat `_silver_loaded_at` pentru audit
- Scris ca tabele Delta în `banking.silver.ref_*`

**Următorul pas:** Notebook `02b_silver_dimensions` — dimensiunile principale cu **SCD Type 2** pentru `dim_customers`. Acolo începe distracția: validări complexe, quarantine, istoricul modificărilor.